# WoundMap AI: Phase 4 — Baseline Model Training (U-Net RGB)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lokeshkuma31/WoundMapAI/blob/main/notebooks/02_baseline_training.ipynb)

**Project**: WoundMap AI (Biophysics & Biomedical Computer-Vision Research Prototype)  
**Governing Blueprint**: `WoundMapAI_Antigravity_Autonomous_Execution_Blueprint.pdf`  
**Experiment**: Baseline Standard sRGB Segmentation (`RUN-01`)  
**Objective**: End-to-end self-contained training and validation of the baseline U-Net on WoundTissue multi-class annotations.

## Step 1: Hardware & Environment Verification
Checks GPU availability (e.g. Tesla T4 on Colab) and logs compute device specifications.

In [ ]:
!nvidia-smi

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if device.type == "cuda":
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Repository Clone & Dependencies

In [ ]:
import os, sys
# In Google Colab, clone repository if not already present
if not os.path.exists("WoundMapAI") and not os.path.exists("src"):
    !git clone https://github.com/Lokeshkuma31/WoundMapAI.git
    %cd WoundMapAI
    !pip install -r requirements.txt

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())

## Step 3: Raw Dataset Acquisition & Verification
Pulls the audited `akabircs/WoundTissue` release into `data/raw/`.

In [ ]:
raw_dir = os.path.join("data", "raw", "WoundTissue")
if not os.path.exists(raw_dir):
    !git clone https://github.com/akabircs/WoundTissue.git "data/raw/WoundTissue"

import pandas as pd
train_df = pd.read_csv("data/manifests/train.csv")
val_df = pd.read_csv("data/manifests/val.csv")
print(f"Verified manifests: Train={len(train_df)} images, Val={len(val_df)} images")

## Step 4: Model Instantiation & Loss Configuration
We instantiate the modular U-Net (3 input channels, 7 classes) and the Composite Loss ($0.5 \times \text{Dice} + 0.5 \times \text{CE}$).

In [ ]:
import src.woundmap.compat
from src.woundmap.models import build_model
from src.woundmap.train import CompositeLoss
from src.woundmap.data import build_dataloader

model = build_model("unet", in_channels=3, num_classes=7, base_features=32).to(device)
criterion = CompositeLoss(dice_weight=0.5, ce_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

train_loader = build_dataloader(
    manifest_csv="data/manifests/train.csv",
    batch_size=4,
    shuffle=True,
    target_size=(512, 512),
    color_space="RGB",
    illumination_norm="none",
    augment=True,
    seed=42
)

val_loader = build_dataloader(
    manifest_csv="data/manifests/val.csv",
    batch_size=2,
    shuffle=False,
    target_size=(512, 512),
    color_space="RGB",
    illumination_norm="none",
    augment=False,
    seed=42
)

print(f"Initialized U-Net with {sum(p.numel() for p in model.parameters()):,} parameters.")

## Step 5: Execute Training Loop with Checkpointing

In [ ]:
from src.woundmap.train import train_model

save_dir = "outputs/checkpoints/baseline"
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=50,
    patience=10,
    save_dir=save_dir,
    model_name="best_baseline_unet_rgb.pt"
)

print("Best Epoch:", history["best_epoch"], "Best Val Macro Dice:", history["best_val_macro_dice"])

## Step 6: Learning Curves & Qualitative Validation Overlays

In [ ]:
import numpy as np
from PIL import Image
from src.woundmap.masks import mask_to_rgb, WOUNDTISSUE_PALETTE

# Load best checkpoint for validation prediction
ckpt = torch.load(os.path.join(save_dir, "best_baseline_unet_rgb.pt"), map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

batch = next(iter(val_loader))
images = batch["image"].to(device)
masks = batch["mask"].to(device)
ids = batch["image_id"]

with torch.no_grad():
    logits = model(images)
    preds = torch.argmax(logits, dim=1).cpu().numpy()

print(f"Predictions generated for {len(ids)} validation images. Class predictions range: {np.unique(preds)}")